# Calculus and Gradient Descent for Machine Learning

**What is Calculus in ML?**  
Calculus is how ML models learn. Every time a neural network updates its weights, it uses calculus — specifically derivatives — to figure out which direction to move to reduce error.

**Why does it matter?**  
- Gradient descent is the algorithm that trains every ML model
- Backpropagation in neural networks is just the chain rule from calculus
- Every optimizer — SGD, Adam, RMSProp — is a variation of gradient descent
- Loss functions and how we minimize them come directly from calculus

**What you'll learn:**
- Derivatives — what they are and how to compute them
- Partial derivatives
- Chain rule
- Gradient descent from scratch
- Learning rate and its effect
- Variants of gradient descent
- Backpropagation intuition

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Derivatives

A derivative measures how much a function's output changes when you change its input by a tiny amount.

**Formula:** f'(x) = lim(h->0) [ f(x+h) - f(x) ] / h

In ML terms: if f is the loss function and x is a weight, the derivative tells you how much the loss changes when you nudge that weight. This is exactly what we need to know to reduce the loss.

In [ ]:
# Numerical derivative — computing it from definition
def numerical_derivative(f, x, h=1e-5):
    return (f(x + h) - f(x)) / h

# Example function: f(x) = x^2
# Analytical derivative: f'(x) = 2x
f = lambda x: x**2

x = 3.0
print('f(x) = x^2 at x=3')
print('Numerical derivative:', numerical_derivative(f, x))  # should be ~6
print('Analytical derivative (2x):', 2*x)                  # exactly 6

# Common derivatives you need to know in ML
print('\nCommon derivatives:')
print('d/dx [x^2] = 2x')
print('d/dx [x^3] = 3x^2')
print('d/dx [e^x] = e^x')
print('d/dx [ln(x)] = 1/x')
print('d/dx [sigmoid(x)] = sigmoid(x) * (1 - sigmoid(x))')

In [ ]:
# Visualise what a derivative means
x = np.linspace(-3, 3, 100)
y = x**2

# Tangent line at x=1 — slope is the derivative = 2*1 = 2
x0 = 1.0
slope = 2 * x0
tangent = slope * (x - x0) + x0**2

plt.figure(figsize=(7, 4))
plt.plot(x, y, label='f(x) = x^2', color='blue')
plt.plot(x, tangent, '--', label=f'Tangent at x=1 (slope={slope})', color='red')
plt.scatter([x0], [x0**2], color='red', zorder=5)
plt.ylim(-1, 9)
plt.legend()
plt.title('Derivative = slope of tangent line at a point')
plt.grid(True, alpha=0.3)
plt.show()

print('The slope tells us: if we increase x by 1, f(x) increases by ~2')
print('In ML: if we increase a weight by 1, loss changes by ~slope')

## 2. Activation Function Derivatives

These are the derivatives you will actually use in neural networks during backpropagation.

In [ ]:
x = np.linspace(-6, 6, 200)

# Sigmoid and its derivative
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

# ReLU and its derivative
def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)   # 1 if x>0, else 0

# Tanh and its derivative
def tanh(x):
    return np.tanh(x)

def tanh_derivative(x):
    return 1 - np.tanh(x)**2

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, func, deriv, name, color in zip(
    axes,
    [sigmoid, relu, tanh],
    [sigmoid_derivative, relu_derivative, tanh_derivative],
    ['Sigmoid', 'ReLU', 'Tanh'],
    ['blue', 'green', 'purple']
):
    ax.plot(x, func(x), color=color, label=name, linewidth=2)
    ax.plot(x, deriv(x), '--', color=color, alpha=0.6, label=f'{name} derivative')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.legend(fontsize=8)
    ax.set_title(name)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-1.5, 1.5)

plt.suptitle('Activation Functions and Their Derivatives')
plt.tight_layout()
plt.show()

## 3. Partial Derivatives

When a function has multiple inputs (like a loss function with thousands of weights), we use partial derivatives — the derivative with respect to one variable while holding all others constant.

In ML: the loss function depends on all weights. We compute the partial derivative with respect to each weight to know how to update it.

In [ ]:
# Example: f(w1, w2) = w1^2 + 2*w1*w2 + w2^2
# Partial derivative w.r.t w1: df/dw1 = 2*w1 + 2*w2
# Partial derivative w.r.t w2: df/dw2 = 2*w1 + 2*w2

def f(w1, w2):
    return w1**2 + 2*w1*w2 + w2**2

def partial_w1(w1, w2):
    return 2*w1 + 2*w2

def partial_w2(w1, w2):
    return 2*w1 + 2*w2

w1, w2 = 3.0, 2.0
print(f'f({w1}, {w2}) = {f(w1, w2)}')
print(f'df/dw1 at ({w1},{w2}) = {partial_w1(w1, w2)}')
print(f'df/dw2 at ({w1},{w2}) = {partial_w2(w1, w2)}')

# The gradient is just the vector of all partial derivatives
gradient = np.array([partial_w1(w1, w2), partial_w2(w1, w2)])
print(f'\nGradient vector: {gradient}')
print('This vector points in the direction of steepest increase')
print('To minimise loss, we move in the OPPOSITE direction')

## 4. Chain Rule

When functions are composed (one inside another), the chain rule tells us how to compute the derivative.

**Formula:** if y = f(g(x)), then dy/dx = f'(g(x)) * g'(x)

This is exactly what backpropagation is — applying the chain rule layer by layer through a neural network.

In [ ]:
# Example: y = sigmoid(w*x + b)
# This is literally one neuron in a neural network

# Forward pass
x = 2.0
w = 0.5
b = 0.1

z = w * x + b           # linear step
y = sigmoid(z)          # activation step
print('Forward pass:')
print(f'  z = w*x + b = {z}')
print(f'  y = sigmoid(z) = {y:.4f}')

# Backward pass using chain rule
# dy/dw = dy/dz * dz/dw
# dy/dz = sigmoid_derivative(z)
# dz/dw = x

dy_dz = sigmoid_derivative(z)   # derivative of sigmoid
dz_dw = x                       # derivative of z w.r.t w
dy_dw = dy_dz * dz_dw           # chain rule

dz_db = 1                       # derivative of z w.r.t b
dy_db = dy_dz * dz_db

print('\nBackward pass (chain rule):')
print(f'  dy/dz = sigmoid_derivative(z) = {dy_dz:.4f}')
print(f'  dz/dw = x = {dz_dw}')
print(f'  dy/dw = dy/dz * dz/dw = {dy_dw:.4f}')
print(f'  dy/db = dy/dz * dz/db = {dy_db:.4f}')
print('\nThese gradients tell us how to update w and b to reduce loss')

## 5. Gradient Descent

Gradient descent is the algorithm that trains ML models. It iteratively moves weights in the direction that reduces the loss.

**Update rule:** w = w - learning_rate * gradient

Why subtract? Because the gradient points uphill. We want to go downhill to reduce loss.

In [ ]:
# Gradient descent on a simple function f(x) = x^2
# Minimum is at x=0, f(0)=0
# Derivative: f'(x) = 2x

def gradient_descent(start, learning_rate, n_iterations):
    x = start
    history = [x]

    for _ in range(n_iterations):
        gradient = 2 * x          # derivative of x^2
        x = x - learning_rate * gradient
        history.append(x)

    return x, history

final_x, history = gradient_descent(start=8.0, learning_rate=0.1, n_iterations=30)

print(f'Started at x = 8.0')
print(f'Final x = {final_x:.6f}  (true minimum = 0)')
print(f'Final f(x) = {final_x**2:.6f}')

# Visualise convergence
x_range = np.linspace(-9, 9, 200)
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(x_range, x_range**2, color='blue', label='f(x) = x^2')
plt.scatter(history, [x**2 for x in history],
            c=range(len(history)), cmap='Reds', zorder=5, label='Steps')
plt.title('Gradient Descent Steps')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot([x**2 for x in history], color='red')
plt.xlabel('Iteration')
plt.ylabel('f(x) — Loss')
plt.title('Loss Curve')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Learning Rate

The learning rate controls how big each step is. Too large — overshoots and diverges. Too small — takes forever to converge. Choosing the right learning rate is one of the most important decisions in training ML models.

In [ ]:
learning_rates = [0.01, 0.1, 0.5, 1.1]
colors = ['blue', 'green', 'orange', 'red']
labels = ['Too slow (0.01)', 'Good (0.1)', 'Fast (0.5)', 'Diverges (1.1)']

plt.figure(figsize=(12, 4))

for lr, color, label in zip(learning_rates, colors, labels):
    _, history = gradient_descent(start=8.0, learning_rate=lr, n_iterations=30)
    losses = [x**2 for x in history]
    # clip for visualisation
    losses = np.clip(losses, 0, 100)
    plt.plot(losses, color=color, label=label, linewidth=2)

plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Effect of Learning Rate on Convergence')
plt.legend()
plt.ylim(0, 100)
plt.grid(True, alpha=0.3)
plt.show()

## 7. Variants of Gradient Descent

| Variant | Uses | Pro | Con |
|---------|------|-----|-----|
| Batch GD | Entire dataset per step | Stable | Slow on large data |
| Stochastic GD | 1 sample per step | Fast | Noisy, unstable |
| Mini-batch GD | Small batch per step | Best of both | Need to tune batch size |

In [ ]:
np.random.seed(42)

# Generate dataset: y = 3x + 2 + noise
X = np.random.rand(100) * 10
y = 3 * X + 2 + np.random.randn(100) * 2

def mse_loss(X, y, w, b):
    predictions = w * X + b
    return np.mean((predictions - y)**2)

def gradients(X, y, w, b):
    n = len(X)
    predictions = w * X + b
    errors = predictions - y
    dw = (2/n) * np.dot(errors, X)
    db = (2/n) * np.sum(errors)
    return dw, db

# Batch Gradient Descent — uses all data every step
def batch_gd(X, y, lr=0.001, epochs=100):
    w, b = 0.0, 0.0
    losses = []
    for _ in range(epochs):
        dw, db = gradients(X, y, w, b)
        w -= lr * dw
        b -= lr * db
        losses.append(mse_loss(X, y, w, b))
    return w, b, losses

# Stochastic Gradient Descent — uses 1 sample per step
def sgd(X, y, lr=0.001, epochs=100):
    w, b = 0.0, 0.0
    losses = []
    for _ in range(epochs):
        idx = np.random.randint(0, len(X))
        xi, yi = X[idx:idx+1], y[idx:idx+1]
        dw, db = gradients(xi, yi, w, b)
        w -= lr * dw
        b -= lr * db
        losses.append(mse_loss(X, y, w, b))
    return w, b, losses

# Mini-batch Gradient Descent — uses small batches
def mini_batch_gd(X, y, lr=0.001, epochs=100, batch_size=16):
    w, b = 0.0, 0.0
    losses = []
    for _ in range(epochs):
        idx = np.random.choice(len(X), batch_size, replace=False)
        xi, yi = X[idx], y[idx]
        dw, db = gradients(xi, yi, w, b)
        w -= lr * dw
        b -= lr * db
        losses.append(mse_loss(X, y, w, b))
    return w, b, losses

w_b, b_b, losses_b = batch_gd(X, y)
w_s, b_s, losses_s = sgd(X, y)
w_m, b_m, losses_m = mini_batch_gd(X, y)

print(f'True:       w=3.0,  b=2.0')
print(f'Batch GD:   w={w_b:.2f}, b={b_b:.2f}')
print(f'SGD:        w={w_s:.2f}, b={b_s:.2f}')
print(f'Mini-batch: w={w_m:.2f}, b={b_m:.2f}')

plt.figure(figsize=(8, 4))
plt.plot(losses_b, label='Batch GD', color='blue')
plt.plot(losses_s, label='SGD', color='red', alpha=0.7)
plt.plot(losses_m, label='Mini-batch GD', color='green')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Batch vs SGD vs Mini-batch')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 8. Backpropagation Intuition

Backpropagation is just gradient descent applied to a neural network using the chain rule. It works backwards from the loss through each layer, computing gradients along the way.

In [ ]:
# One complete forward + backward pass through a 2-layer neural network
np.random.seed(42)

# Network: input(2) -> hidden(3) -> output(1)
# One training sample
x = np.array([0.5, 0.8])       # input
y_true = np.array([1.0])        # true label

# Random weights
W1 = np.random.randn(2, 3)     # (input, hidden)
b1 = np.zeros(3)
W2 = np.random.randn(3, 1)     # (hidden, output)
b2 = np.zeros(1)

lr = 0.1

# Forward pass
z1 = x @ W1 + b1               # (3,)
a1 = sigmoid(z1)                # (3,) — hidden layer output
z2 = a1 @ W2 + b2              # (1,)
a2 = sigmoid(z2)                # (1,) — final prediction

# Loss — MSE
loss = np.mean((a2 - y_true)**2)

print('Forward pass:')
print(f'  Prediction: {a2[0]:.4f}')
print(f'  True label: {y_true[0]}')
print(f'  Loss: {loss:.4f}')

# Backward pass — chain rule all the way back
# Output layer gradients
d_loss  = 2 * (a2 - y_true) / len(y_true)     # dL/da2
d_a2    = sigmoid_derivative(z2)               # da2/dz2
delta2  = d_loss * d_a2                        # dL/dz2

dW2 = a1.reshape(-1, 1) @ delta2.reshape(1, -1)  # dL/dW2
db2 = delta2

# Hidden layer gradients
d_a1    = sigmoid_derivative(z1)               # da1/dz1
delta1  = (delta2 @ W2.T) * d_a1              # dL/dz1

dW1 = x.reshape(-1, 1) @ delta1.reshape(1, -1)   # dL/dW1
db1 = delta1

# Weight update
W2 -= lr * dW2
b2 -= lr * db2
W1 -= lr * dW1
b1 -= lr * db1

# Forward pass again to check improvement
z1_new = x @ W1 + b1
a1_new = sigmoid(z1_new)
z2_new = a1_new @ W2 + b2
a2_new = sigmoid(z2_new)
loss_new = np.mean((a2_new - y_true)**2)

print('\nAfter one weight update:')
print(f'  New prediction: {a2_new[0]:.4f}  (closer to {y_true[0]})')
print(f'  New loss: {loss_new:.4f}  (was {loss:.4f})')
print(f'  Loss reduced: {loss - loss_new:.4f}')

---

## Mini Project — Linear Regression Trained with Gradient Descent

**Task:** Train a linear regression model from scratch using gradient descent — no sklearn, no normal equation. Just calculus.

In [ ]:
np.random.seed(42)

# Dataset: house size vs price
# size in hundreds of sqft, price in lakhs
X = np.random.rand(100) * 30 + 10      # size: 10 to 40
y = 2.5 * X + 5 + np.random.randn(100) * 3   # price = 2.5*size + 5 + noise

# Normalise X for stable training
X_norm = (X - X.mean()) / X.std()

print('Dataset:')
print(f'  Samples: {len(X)}')
print(f'  X range: {X.min():.1f} to {X.max():.1f}')
print(f'  y range: {y.min():.1f} to {y.max():.1f}')

In [ ]:
# Train with gradient descent
w = 0.0     # weight (slope)
b = 0.0     # bias (intercept)
lr = 0.01
epochs = 200
losses = []

for epoch in range(epochs):
    # Forward pass
    y_pred = w * X_norm + b

    # Loss — MSE
    loss = np.mean((y_pred - y)**2)
    losses.append(loss)

    # Gradients
    dw = (2/len(X)) * np.dot((y_pred - y), X_norm)
    db = (2/len(X)) * np.sum(y_pred - y)

    # Update weights
    w -= lr * dw
    b -= lr * db

    if epoch % 50 == 0:
        print(f'Epoch {epoch:3d} | Loss: {loss:.4f} | w: {w:.4f} | b: {b:.4f}')

print(f'\nFinal | Loss: {losses[-1]:.4f} | w: {w:.4f} | b: {b:.4f}')

In [ ]:
# Visualise results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Predictions vs actual
y_pred_final = w * X_norm + b
ax1.scatter(X, y, alpha=0.5, label='Actual data')
ax1.scatter(X, y_pred_final, alpha=0.5, color='red', label='Predictions', s=10)
sort_idx = np.argsort(X)
ax1.plot(X[sort_idx], y_pred_final[sort_idx], color='red', linewidth=2)
ax1.set_xlabel('House size')
ax1.set_ylabel('Price')
ax1.set_title('Linear Regression via Gradient Descent')
ax1.legend()

# Loss curve
ax2.plot(losses, color='blue')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MSE Loss')
ax2.set_title('Loss Curve')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## Common Mistakes to Avoid

1. **Learning rate too high** — loss explodes or oscillates instead of decreasing. Always start with 0.001 or 0.01 and adjust.
2. **Not normalising features** — gradient descent converges much slower on unnormalised data. Always normalise before training.
3. **Not tracking the loss curve** — always plot loss vs epochs. If loss is not decreasing, something is wrong with gradients or learning rate.
4. **Vanishing gradients** — sigmoid and tanh derivatives become very small for large inputs. This kills learning in deep networks. ReLU fixes this.
5. **Forgetting the chain rule** — when computing gradients manually, always work backwards from the loss and apply the chain rule at every step.

---

## Interview Questions

1. What is a derivative and what does it tell us in the context of ML?
2. What is gradient descent and why do we subtract the gradient?
3. What happens if the learning rate is too high or too low?
4. What is the difference between batch, stochastic, and mini-batch gradient descent?
5. What is backpropagation and how does the chain rule apply to it?
6. What is the vanishing gradient problem and which activation functions cause it?
7. Why do we normalise features before training?

---

**Math section complete.**  
**Next:** `02_python_for_ml/01_numpy.ipynb` — now that you understand the math, NumPy will make much more sense.